In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import faiss

In [78]:
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4302.49it/s]


In [121]:
with open("500DaysofSummer.txt", "r", encoding="utf-8") as file:
    text = file.read()

def chunk_text(text, chunk_size=800, overlap=150):
    chunks = []

    for i in range(0, len(text), chunk_size - overlap):
        chunks.append(text[i:i+chunk_size])

    return chunks

chunks = chunk_text(text)

emb = model.encode(chunks)
emb = emb.astype("float32")

index = faiss.IndexFlatL2(emb.shape[1])
index.add(emb)

In [124]:
import re

def clean_text(text):

    # Remove revision stamps
    text = re.sub(r'Yellow Revised.*', '', text)

    # Remove standalone page numbers like 23, 104, 187A
    text = re.sub(r'^\s*\d+[A-Z]?\s*$', '', text, flags=re.MULTILINE)

    # Remove screenplay page markers like (238), (403 1/2)
    text = re.sub(r'\(\d+(?:\s*1/2)?\)', '', text)

    # Remove repeated blank lines
    text = re.sub(r'\n{3,}', '\n\n', text)

    # Remove extra spaces
    text = re.sub(r'[ \t]+', ' ', text)

    return text.strip()

cleaned_text = clean_text(text)
print(len(text))
print(len(cleaned_text))

102103
98366


In [132]:
import re

def chunk_text(text):

    # Find every scene heading
    pattern = r'(?=^(?:INT\.|EXT\.|INT/EXT\.|EXT/INT\.))'

    scenes = re.split(pattern, text, flags=re.MULTILINE)

    chunks = []

    for scene in scenes:
        scene = scene.strip()

        if len(scene) > 100:
            chunks.append(scene)

    return chunks
chunks = chunk_text(cleaned_text)

print(len(chunks))

11


In [133]:
for i in range(5):
    print("="*80)
    print(chunks[i][:400])

DAYS OF SUMMER
by
Scott Neustadter
&
Michael H. Weber
April 16, 2008NOTE: THE FOLLOWING IS A WORK OF FICTION. ANY RESEMBLANCE TO
PERSONS LIVING OR DEAD IS PURELY COINCIDENTAL.ESPECIALLY YOU JENNY BECKMAN.BITCH.FADE IN:
A single number in parenthesis, exactly like so:
EXT. ANGELUS PLAZA — DOWNTOWN LOS ANGELES, CA — DAY 1
And we’re looking at a MAN (20s) and a WOMAN (20s) ona
bench, high above the city of Los Angeles. Their names are
TOM and SUMMER and right now neither one says a word.
CLOSE ON their HANDS, intertwined. Notice the wedding ring
on her finger. CLOSE ON Tom, looking at Summer the way every
woman wants to be looked at.
And then a DISTINGUISHED VOIC
INT. CUBICLE - SAME 6A
Summer answers a call, takes a message, and walks out of her
cubicle down a long narrow hallway.
NARRATOR
Tom meets Summer on June the 8th.
He knows almost immediately...
she’s who he’s been searching for.
CU Summer opening the door to the boardroom, about to come
face to face with Tom for the first time.
N

In [81]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

In [82]:
from groq import Groq

client = Groq(api_key=api_key)

In [134]:
query = "What did Tom buy from IKEA"
query_embedding = model.encode(query)
query_embedding = query_embedding.reshape(1, -1)
distances, indices = index.search(query_embedding, 3)
retrieved_chunks = []

for idx in indices[0]:
    retrieved_chunks.append(chunks[idx])

context = "\n\n".join(retrieved_chunks)

distances, indices = index.search(query_embedding, 10)
print(retrieved_chunks)

IndexError: list index out of range

In [97]:
prompt = f"""
You are a question-answering assistant.

Use ONLY the provided context.

Rules:

1. Never use outside knowledge.
2. Never infer information that is not explicitly stated.
3. If the answer is missing, reply exactly:
   "No information available in the provided context."
4. After every answer, include the exact sentence(s) from the context that support your answer.
5. If no supporting sentence exists, return only:
   "No information available in the provided context."

Context:
{context}
Make sure to answer the question in a compassionate style, as this is about a movie called 500 Days of Summer.

Question:
{query}

"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)
print("Question:" ,query)
print("Answer:" ,response.choices[0].message.content)

Question: What did Tom buy from IKEA
Answer: No information available in the provided context.

There are no sentences in the provided context that mention Tom buying anything from IKEA. The context only shows Tom and Summer browsing the IKEA store, looking at various items, but it does not mention any purchases.
